# Fine-tuning Llama 3.1 8B for Vector-LLM (Colab)

This notebook fine-tunes **Llama 3.1 8B Instruct** using **Unsloth (QLoRA 4-bit)** on a Google Colab T4 or A100 GPU,
then automatically merges and exports the fine-tuned model into **GGUF format (`q4_k_m`)** for Ollama.

### Prerequisites:
- Runtime: Make sure **GPU** is selected (**Runtime > Change runtime type > T4 GPU**).

In [ ]:
# 1. Check GPU
!nvidia-smi

In [ ]:
# 2. Install Unsloth and training dependencies
%%capture
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets transformers

In [ ]:
# 3. Create training dataset generator
import os, json, random

os.makedirs("./tools/data", exist_ok=True)

INTENT_PATTERNS = [
    ("show me monthly revenue as a bar chart", {"intent": "dashboard", "confidence": 0.95, "refined_query": "Show monthly revenue as a bar chart"}),
    ("should we expand sales team in Q4?", {"intent": "decision", "confidence": 0.90, "refined_query": "Should we expand sales team in Q4?"}),
    ("what was total profit in March?", {"intent": "question", "confidence": 0.95, "refined_query": "What was total profit in March?"}),
    ("forecast customer churn for next month", {"intent": "forecast", "confidence": 0.90, "refined_query": "Forecast customer churn for next month"}),
    ("give me an overview of Q2 sales performance", {"intent": "summary", "confidence": 0.92, "refined_query": "Give an overview of Q2 sales performance"}),
    ("plot product unit sales per region", {"intent": "dashboard", "confidence": 0.96, "refined_query": "Plot product unit sales per region"}),
    ("is our marketing spend yielding positive ROI?", {"intent": "decision", "confidence": 0.92, "refined_query": "Is marketing spend yielding positive ROI?"}),
    ("summarize operational costs across departments", {"intent": "summary", "confidence": 0.94, "refined_query": "Summarize operational costs across departments"}),
    ("predict inventory shortage risks for December", {"intent": "forecast", "confidence": 0.91, "refined_query": "Predict inventory shortage risks for December"}),
]

samples = []
for _ in range(300):
    q, expected = random.choice(INTENT_PATTERNS)
    samples.append({
        "instruction": "Analyze the user query and classify intent into one of: dashboard, decision, question, forecast, summary. Respond strictly with JSON.",
        "input": q,
        "output": json.dumps(expected)
    })

train_path = "./tools/data/vector_llm_train.jsonl"
with open(train_path, "w", encoding="utf-8") as f:
    for s in samples:
        f.write(json.dumps(s) + "\n")

print(f"Generated {len(samples)} training samples at {train_path}")

In [ ]:
# 4. Fine-tune Llama 3.1 8B with Unsloth QLoRA
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset

max_seq_length = 2048
base_model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"

print("Loading model:", base_model_name)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = base_model_name,
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)

dataset = load_dataset("json", data_files=train_path, split="train")

def format_prompts(batch):
    instructions = batch["instruction"]
    inputs       = batch["input"]
    outputs      = batch["output"]
    texts = []
    for inst, inp, out in zip(instructions, inputs, outputs):
        text = f"### Instruction:\n{inst}\n\n### Input:\n{inp}\n\n### Response:\n{out}"
        texts.append(text)
    return { "text" : texts }

formatted_dataset = dataset.map(format_prompts, batched=True)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = formatted_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = True,
        logging_steps = 1,
        output_dir = "./vector_llm_output",
    ),
)

trainer.train()

In [ ]:
# 5. Save & Export directly to GGUF (q4_k_m) for Ollama
gguf_dir = "./vector_llm_gguf"
print(f"Exporting fine-tuned model to GGUF at {gguf_dir}...")
model.save_pretrained_gguf(gguf_dir, tokenizer, quantization_method="q4_k_m")
print("Export complete!")

In [ ]:
# 6. Download the GGUF file to your local computer
import glob, os
from google.colab import files

gguf_files = glob.glob(f"{gguf_dir}/**/*.gguf", recursive=True)
if not gguf_files:
    gguf_files = glob.glob(f"{gguf_dir}/*.gguf")

if gguf_files:
    target_file = gguf_files[0]
    print(f"Downloading {target_file}...")
    files.download(target_file)
else:
    print("No .gguf file found. Check contents of gguf_dir:", os.listdir(gguf_dir))

## Next Steps (On Your Local Machine):
1. Move the downloaded `.gguf` file to your `vector-llm/tools/` folder.
2. In `tools/Modelfile`, ensure the `FROM` line points to your `.gguf` file name:
   ```dockerfile
   FROM ./Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf
   ```
3. Create the model in Ollama:
   ```powershell
   ollama create vector-llm:v1 -f tools/Modelfile
   ```
4. In `.env`, set `OLLAMA_MODEL=vector-llm:v1` and restart your FastAPI server!